# ClickHouse tick-data inspection

Load the collected Kalshi tick data from ClickHouse and check its **format** — the tables, their schema,
row counts, and sample rows. Use this to confirm the collector is writing what you expect before you
backtest on it.

Prereqs: ClickHouse is up (`make ch-server`) and a collector has run. Set `CLICKHOUSE_URL` if it isn't at
`http://localhost:8123`.

In [1]:
import os, io
import pandas as pd
import requests

CH = os.environ.get('CLICKHOUSE_URL', 'http://localhost:8123')

def ch_df(sql: str) -> pd.DataFrame:
    """Run a ClickHouse SQL query over HTTP and return a pandas DataFrame."""
    r = requests.post(CH, params={'query': sql + ' FORMAT TabSeparatedWithNames'}, timeout=30)
    r.raise_for_status()
    if not r.text.strip():
        return pd.DataFrame()
    return pd.read_csv(io.StringIO(r.text), sep='\t')

print('ClickHouse', ch_df('SELECT version() AS v').iloc[0]['v'], 'at', CH)

ClickHouse 26.6.1.731 at http://localhost:8123


## 1. Tables in the `kalshi` database

In [2]:
ch_df("SELECT name, total_rows, formatReadableSize(total_bytes) AS size "
      "FROM system.tables WHERE database = 'kalshi' ORDER BY name")

,name,total_rows,size
0,book_features_1s,0,0.00 B
1,orderbook_deltas,100441,2.59 MiB
2,trades,19,15.64 KiB


## 2. Schema of `orderbook_deltas` (the tick table)
Every row is one book event. `is_snapshot=1` marks the first row of a full-book snapshot (resets the
book). The book is **YES-native**: `side=BUY` is a YES bid, `side=SELL` is a YES ask (a NO bid at price q
is stored as a YES ask at 1−q). `price` is dollars in [0.01, 0.99].

In [3]:
ch_df('DESCRIBE TABLE kalshi.orderbook_deltas')

,name,type,default_type,default_expression,comment,codec_expression,ttl_expression
0,timestamp,"DateTime64(9, \'UTC\')",NaN,NaN,NaN,NaN,NaN
1,instrument_id,LowCardinality(String),NaN,NaN,NaN,NaN,NaN
2,venue,LowCardinality(String),DEFAULT,\'KALSHI\',NaN,NaN,NaN
3,action,"Enum8(\'ADD\' = 1, \'UPDATE\' = 2, \'DELETE\' ...",NaN,NaN,NaN,NaN,NaN
4,side,"Enum8(\'BUY\' = 1, \'SELL\' = 2)",NaN,NaN,NaN,NaN,NaN
5,price,Float64,NaN,NaN,NaN,NaN,NaN
6,size,Float64,NaN,NaN,NaN,NaN,NaN
7,market_alias,LowCardinality(String),DEFAULT,\'\',NaN,NaN,NaN
8,sequence,Int64,DEFAULT,0,NaN,NaN,NaN
9,is_snapshot,UInt8,DEFAULT,0,NaN,NaN,NaN


In [4]:
ch_df('DESCRIBE TABLE kalshi.trades')

,name,type,default_type,default_expression,comment,codec_expression,ttl_expression
0,ts_event,"DateTime64(9, \'UTC\')",NaN,NaN,NaN,NaN,NaN
1,instrument_id,LowCardinality(String),NaN,NaN,NaN,NaN,NaN
2,venue,LowCardinality(String),DEFAULT,\'KALSHI\',NaN,NaN,NaN
3,aggressor_side,"Enum8(\'yes\' = 1, \'no\' = 2)",NaN,NaN,NaN,NaN,NaN
4,price,Float64,NaN,NaN,NaN,NaN,NaN
5,size,Float64,NaN,NaN,NaN,NaN,NaN
6,market_alias,LowCardinality(String),DEFAULT,\'\',NaN,NaN,NaN
7,trade_id,String,DEFAULT,\'\',NaN,NaN,NaN


## 3. Coverage: how much, how many instruments, what time span

In [ ]:
ch_df("SELECT count() AS deltas, uniqExact(instrument_id) AS instruments, "
      "       min(timestamp) AS first, max(timestamp) AS last, "
      "       countIf(is_snapshot = 1) AS snapshot_rows "
      "FROM kalshi.orderbook_deltas")

In [6]:
# most active instruments
ch_df("SELECT instrument_id, count() AS deltas, min(timestamp) AS first, max(timestamp) AS last "
      "FROM kalshi.orderbook_deltas GROUP BY instrument_id ORDER BY deltas DESC LIMIT 10")

,instrument_id,deltas,first,last
0,KXNATGASD-26JUN1517-T3.085,7236,2026-06-13 03:50:00.086142000,2026-06-13 14:11:18.383000000
1,KXNATGASD-26JUN1517-T2.990,3341,2026-06-13 03:50:01.535746000,2026-06-13 14:11:18.383000000
2,KXNATGASD-26JUN1517-T3.115,3305,2026-06-13 03:49:22.731285000,2026-06-13 14:11:18.391000000
3,KXNATGASD-26JUN1517-T3.120,3246,2026-06-13 03:49:22.657755000,2026-06-13 14:11:19.406000000
4,KXNATGASD-26JUN1517-T3.125,3214,2026-06-13 03:49:22.584866000,2026-06-13 14:11:19.407000000
5,KXNATGASD-26JUN1517-T3.105,3128,2026-06-13 03:49:59.800513000,2026-06-13 14:11:17.361000000
6,KXNATGASD-26JUN1517-T3.130,3024,2026-06-13 03:49:22.511044000,2026-06-13 14:11:19.405000000
7,KXNATGASD-26JUN1517-T2.985,3013,2026-06-13 03:50:01.602753000,2026-06-13 14:11:18.383000000
8,KXNATGASD-26JUN1517-T3.100,2995,2026-06-13 03:49:59.872725000,2026-06-13 14:11:18.394000000
9,KXNATGASD-26JUN1517-T3.090,2945,2026-06-13 03:50:00.019917000,2026-06-13 14:11:18.390000000


## 4. Sample raw rows (what the data actually looks like)

In [7]:
inst = ch_df("SELECT instrument_id FROM kalshi.orderbook_deltas GROUP BY instrument_id "
             "ORDER BY count() DESC LIMIT 1")
inst = inst.iloc[0]['instrument_id'] if len(inst) else None
print('sample instrument:', inst)
ch_df(f"SELECT timestamp, instrument_id, action, side, price, size, sequence, is_snapshot "
      f"FROM kalshi.orderbook_deltas WHERE instrument_id = '{inst}' ORDER BY timestamp, sequence LIMIT 20")

sample instrument: KXNATGASD-26JUN1517-T3.085


,timestamp,instrument_id,action,side,price,size,sequence,is_snapshot
0,2026-06-13 03:50:00.086142000,KXNATGASD-26JUN1517-T3.085,ADD,BUY,0.13,1011.00,0,1
1,2026-06-13 03:50:00.086142000,KXNATGASD-26JUN1517-T3.085,ADD,BUY,0.14,200.00,0,0
2,2026-06-13 03:50:00.086142000,KXNATGASD-26JUN1517-T3.085,ADD,BUY,0.25,531.00,0,0
3,2026-06-13 03:50:00.086142000,KXNATGASD-26JUN1517-T3.085,ADD,BUY,0.26,51.00,0,0
4,2026-06-13 03:50:00.086142000,KXNATGASD-26JUN1517-T3.085,ADD,BUY,0.49,192.00,0,0
5,2026-06-13 03:50:00.086142000,KXNATGASD-26JUN1517-T3.085,ADD,BUY,0.53,60.00,0,0
6,2026-06-13 03:50:00.086142000,KXNATGASD-26JUN1517-T3.085,ADD,BUY,0.59,77.00,0,0
7,2026-06-13 03:50:00.086142000,KXNATGASD-26JUN1517-T3.085,ADD,BUY,0.63,47.00,0,0
8,2026-06-13 03:50:00.086142000,KXNATGASD-26JUN1517-T3.085,ADD,BUY,0.64,288.00,0,0
9,2026-06-13 03:50:00.086142000,KXNATGASD-26JUN1517-T3.085,ADD,BUY,0.65,389.14,0,0


In [8]:
trades = ch_df('SELECT * FROM kalshi.trades ORDER BY ts_event DESC LIMIT 10')
print(f'{len(trades)} recent trades' if len(trades) else 'no trades captured yet (NatGas is often quiet)')
trades

10 recent trades


,ts_event,instrument_id,venue,aggressor_side,price,size,market_alias,trade_id
0,2026-06-13 13:39:53.920000000,KXNATGASD-26JUN1517-T2.985,KALSHI,yes,0,0,NaN,a37bf758-cff4-5355-8d16-cd98a1af6b68
1,2026-06-13 12:26:53.149000000,KXNATGASD-26JUN1517-T2.985,KALSHI,yes,0,0,NaN,93480ded-021c-6ef1-0065-7f19af2ad6f8
2,2026-06-13 12:26:37.230000000,KXNATGASD-26JUN1517-T2.985,KALSHI,yes,0,0,NaN,4c3fb3dc-dc44-711d-eef7-74ec35555b64
3,2026-06-13 12:26:25.735000000,KXNATGASD-26JUN1517-T2.990,KALSHI,yes,0,0,NaN,59a5ebb9-52d6-63c0-0f9f-77aca6df0656
4,2026-06-13 12:26:16.144000000,KXNATGASD-26JUN1517-T2.990,KALSHI,yes,0,0,NaN,c7284dc9-341a-7768-7c8d-edb7607551a6
5,2026-06-13 10:37:26.036000000,KXNATGASD-26JUN1517-T3.120,KALSHI,no,0,0,NaN,f8a1fc7e-9168-655c-7c02-1dafcd4234c1
6,2026-06-13 10:36:07.179000000,KXNATGASD-26JUN1517-T3.120,KALSHI,yes,0,0,NaN,04aa1560-8a5f-69cb-5579-49e02c1cd26b
7,2026-06-13 09:02:13.652000000,KXNATGASD-26JUN1517-T2.985,KALSHI,yes,0,0,NaN,a5799866-ddc0-5136-b415-5e22277c6675
8,2026-06-13 07:02:54.551000000,KXNATGASD-26JUN1517-T3.095,KALSHI,no,0,0,NaN,67f18234-800b-67d3-c89a-c70170100f6f
9,2026-06-13 07:02:54.466000000,KXNATGASD-26JUN1517-T3.090,KALSHI,yes,0,0,NaN,2e39d9cc-f1be-5b07-7826-dd02baf17548


## 5. Reconstruct top-of-book for one instrument
Replay the deltas (snapshot resets + add/update/delete) into a book and show best bid/ask/mid — confirms
the data is internally consistent and ready to backtest on.

In [9]:
d = ch_df(f"SELECT toUnixTimestamp64Nano(timestamp) AS ts_ns, action, side, price, size, is_snapshot "
          f"FROM kalshi.orderbook_deltas WHERE instrument_id = '{inst}' ORDER BY timestamp, sequence")
book = {'BUY': {}, 'SELL': {}}; tob = []
for _, r in d.iterrows():
    if int(r['is_snapshot']): book = {'BUY': {}, 'SELL': {}}
    if r['action'] == 'DELETE' or r['size'] <= 0: book[r['side']].pop(r['price'], None)
    else: book[r['side']][r['price']] = r['size']
    bid = max(book['BUY']) if book['BUY'] else None
    ask = min(book['SELL']) if book['SELL'] else None
    tob.append({'ts_ns': int(r['ts_ns']), 'best_bid': bid, 'best_ask': ask,
                'mid': (bid+ask)/2 if bid and ask else None})
tob = pd.DataFrame(tob)
print(f'{len(tob)} book states for {inst}; last quote: bid={tob.best_bid.iloc[-1]} ask={tob.best_ask.iloc[-1]}')
tob.tail(8)

7236 book states for KXNATGASD-26JUN1517-T3.085; last quote: bid=0.62 ask=0.64


,ts_ns,best_bid,best_ask,mid
7228,1781359705000000000,0.62,0.64,0.63
7229,1781359705000000000,0.62,0.64,0.63
7230,1781359705035000000,0.62,0.64,0.63
7231,1781359705036000000,0.62,0.64,0.63
7232,1781359877000000000,0.62,0.64,0.63
7233,1781359877363000000,0.62,0.64,0.63
7234,1781359878000000000,0.62,0.64,0.63
7235,1781359878383000000,0.62,0.64,0.63


---
**This is exactly the data the backtester consumes** (`--source clickhouse`, or `--source ndjson` for the
`data/raw/*.ndjson.gz` captures of the same schema). See `backtesterDescription.md` for the data contract
and `howToRun.md` to run a backtest on it.